# 单层 skyrmion：OVF 离线测量

仅处理明确选择的文件，零LLG/零求场。默认 Run All 不创建数据、不分析用户路径。升级安装包后请重启并选用对应内核；不要重载正在运行的模拟内核。几何和相位分别检查valid，未知时间或静态不计算速度。


In [ ]:
from pathlib import Path
import uuid
from fimrun.postprocess import analyze_ovf, frames_from_state, generate_demo
rows = []


## 先看解析示例

将 generate_example 改为 True 才生成全新目录。五帧含跨边界、呼吸和反平行B；真速度(10,-3)m/s，B-A=(6,-3)nm，校正相位差0.15rad。只构造数组，不求解动力学。重复执行会选择另一个新目录，不覆盖。


In [ ]:
generate_example = False
if generate_example:
    from fimrun.postprocess import generate_demo as generate
    example_dir = Path.cwd() / ('skyrmion-example-' + uuid.uuid4().hex[:8])
    rows = generate(example_dir)
    print('CSV:', example_dir / 'analysis.csv')


## 明确选择实际输入

自动输出可由 state 的哈希清单配对。旧OVF请改为显式A/B路径和有证据的mapping；不猜时间、PBC、单位或Ms。不同segment断轨；启用时间采样假设意味着你确认每帧实际位移小于半盒，算法无法识别漏掉的完整圈数。静态stage=sdesc只按接受iteration解读。详情见 docs/interface-v01/SKYRMION_POSTPROCESSING.md。


In [ ]:
run_analysis = False
state_path = Path('/replace/with/drive.state')
output_csv = Path('/replace/with/new-analysis.csv')
assert_temporal_sampling = False  # 确认物理采样足够密后才设True
if run_analysis:
    frames = frames_from_state(state_path)
    rows = analyze_ovf(frames, output_csv, 
                       assume_temporal_sampling=assert_temporal_sampling,
                       ab_reference='antiparallel')
    print('CSV:', output_csv, 'rows:', len(rows))


## 检查结果

中心坐标主列折回原盒，轨迹图须使用unwrapped列；速度端点NaN。下面表格显示A/B面积中心、半径、形状、原始相位及有效标志。接近圆时主轴NaN是预期。raw值只作诊断。


In [ ]:
from html import escape
from IPython.display import display, HTML
columns = ['time_s','iteration','A_area_x','A_area_y','B_area_x','B_area_y',
           'A_radius_m','B_radius_m','A_axis_ratio','A_axis_rad','A_chi_rad',
           'AB_chi_difference_rad','A_area_vx_mps','A_area_vy_mps',
           'A_valid_geometry','A_valid_topology','A_valid_helicity','valid_velocity','reason_codes']
if rows:
    table = '<table><tr>' + ''.join('<th>'+escape(c)+'</th>' for c in columns) + '</tr>'
    for row in rows:
        table += '<tr>' + ''.join('<td>'+escape(str(row.get(c, float('nan'))))+'</td>' for c in columns) + '</tr>'
    display(HTML(table+'</table>'))
else:
    print('未选择处理；没有读取用户磁态，也没有写文件。')


## Notebook 原生 SVG 图

仅使用上面明确选择并完成分析的rows；无需matplotlib或新增依赖，不启动浏览器。动态追踪使用unwrapped坐标，未追踪时只画wrapped散点；静态中心始终使用wrapped坐标。半径、形状和相位按真实时间/接受迭代作图，未知静态迭代明确标为input index。无效帧和相位±pi跳变不连线。


In [ ]:
from fimrun.postprocess import summary_svg
from IPython.display import SVG, display
if rows:
    analysis_svg = summary_svg(rows)
    display(SVG(analysis_svg))


训练包络不是通用物理误差保证。原OVF窄壁/高频相位的分辨率仍需研究者判断；更密的轮廓插值不会补回原网格信息。没有跨segment自动拼接或多对象匹配。source_bindings保留实际输入、映射、实现与校准内容哈希，原始数据不改写。
